# Exercices PyTorch — Version complétée

Ce notebook contient les solutions des 4 exercices de l'activité pratique :

1. Régression linéaire multiple (salaires)
2. Classification Iris
3. MNIST avec un réseau dense (MLP)
4. MNIST avec un CNN

Chaque `...` du sujet a été remplacé par le code correspondant, et les questions théoriques ont une réponse en texte.

In [1]:
import torch
from torch import nn
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Appareil :", device)

Appareil : cpu


# Exercice 1 — Régression : prédire le salaire

## 1. Génération des données

In [3]:
torch.manual_seed(42)

n = 300
anciennete = torch.randint(0, 21, (n, 1)).float()
expertise = torch.randint(1, 6, (n, 1)).float()
bruit = torch.randn(n, 1) * 500

salaires = 4000 + 600 * anciennete + 1500 * expertise + bruit

X = torch.cat([anciennete, expertise], dim=1)   # (300, 2)
y = salaires                                    # (300, 1)

for i in range(5):
    print(
        f"Ancienneté : {X[i, 0].item():.0f} ans | "
        f"Expertise : {X[i, 1].item():.0f}/5 | "
        f"Salaire : {y[i, 0].item():.2f} DH"
    )

Ancienneté : 15 ans | Expertise : 2/5 | Salaire : 16262.90 DH
Ancienneté : 2 ans | Expertise : 3/5 | Salaire : 10541.39 DH
Ancienneté : 4 ans | Expertise : 3/5 | Salaire : 10948.37 DH
Ancienneté : 19 ans | Expertise : 3/5 | Salaire : 20028.54 DH
Ancienneté : 18 ans | Expertise : 2/5 | Salaire : 18036.38 DH


**Travail demandé :** dimensions, salaire moyen, nuage de points coloré selon l'expertise.

In [ ]:
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)
print(f"Salaire moyen : {y.mean().item():.2f} DH")

plt.figure(figsize=(7, 5))
sc = plt.scatter(X[:, 0].numpy(), y[:, 0].numpy(), c=X[:, 1].numpy(), cmap="viridis", s=20)
plt.colorbar(sc, label="Niveau d'expertise (1-5)")
plt.xlabel("Ancienneté (années)")
plt.ylabel("Salaire (DH)")
plt.title("Salaire selon l'ancienneté")
plt.grid(alpha=0.3)
plt.show()

## 2. Séparer et préparer les données

In [ ]:
indices = torch.randperm(n)
limite = int(0.8 * n)

train_indices = indices[:limite]
test_indices = indices[limite:]

X_train = X[train_indices]
X_test = X[test_indices]

# Salaires en milliers de DH
y_train = y[train_indices] / 1000
y_test = y[test_indices] / 1000

# Statistiques calculées uniquement sur le jeu d'entraînement
moyenne = X_train.mean(dim=0, keepdim=True)
ecart_type = X_train.std(dim=0, keepdim=True)

X_train_norm = (X_train - moyenne) / ecart_type
X_test_norm = (X_test - moyenne) / ecart_type

## 3. Construire et entraîner le modèle

In [ ]:
# Deux variables d'entrée et une sortie
model = nn.Linear(2, 1)

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

historique = []

model.train()

for epoch in range(500):
    # 1. Réinitialiser les gradients
    optimizer.zero_grad()

    # 2. Prédire les salaires
    predictions = model(X_train_norm)

    # 3. Calculer la perte
    loss = criterion(predictions, y_train)

    # 4. Calculer les gradients
    loss.backward()

    # 5. Mettre à jour les paramètres
    optimizer.step()

    historique.append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"Époque {epoch + 1} | Perte : {loss.item():.4f}")

In [ ]:
plt.plot(historique)
plt.xlabel("Époque")
plt.ylabel("MSE — salaires en milliers de DH")
plt.title("Évolution de la perte d'entraînement")
plt.grid()
plt.show()

## 4. Évaluer les prédictions

In [ ]:
model.eval()

with torch.no_grad():
    predictions_test = model(X_test_norm)

    # Conversion en DH
    predictions_dh = predictions_test * 1000
    salaires_reels_dh = y_test * 1000

    mae = torch.abs(predictions_dh - salaires_reels_dh).mean()

print(f"Erreur absolue moyenne : {mae.item():.2f} DH")

## 5. Prédire le salaire d'un nouveau salarié

In [ ]:
nouveau_salarie = torch.tensor([[7.0, 4.0]])

# Même normalisation qu'à l'entraînement
nouveau_salarie_norm = (nouveau_salarie - moyenne) / ecart_type

with torch.no_grad():
    salaire_predit = model(nouveau_salarie_norm) * 1000   # retour en DH

print(f"Salaire estimé : {salaire_predit.item():.2f} DH")
print("Valeur théorique sans bruit : 14200 DH")

**Questions complémentaires — retrouver les coefficients en unités d'origine**

In [ ]:
w_norm = model.weight.detach()   # (1, 2)
b_norm = model.bias.detach()     # (1,)

# y = w·(x - mu)/sigma + b  =>  y = (w/sigma)·x + (b - sum(w*mu/sigma))
w_orig = (w_norm / ecart_type) * 1000
b_orig = (b_norm - (w_norm * moyenne / ecart_type).sum()) * 1000

print(f"Effet d'une année d'ancienneté : {w_orig[0, 0].item():.1f} DH   (attendu ≈ 600)")
print(f"Effet d'un niveau d'expertise  : {w_orig[0, 1].item():.1f} DH   (attendu ≈ 1500)")
print(f"Salaire de base                : {b_orig.item():.1f} DH   (attendu ≈ 4000)")

**Réponses aux questions**

- **Effet d'une année d'ancienneté (expertise constante) :** environ **+600 DH** de salaire mensuel, c'est le coefficient `w₁` retrouvé ci-dessus.
- **Pourquoi la perte n'est-elle pas nulle ?** Les salaires contiennent un bruit aléatoire d'écart-type 500 DH, soit 0,5 en milliers de DH. Aucun modèle ne peut prédire ce bruit, donc la MSE minimale est proche de 0,5² = **0,25**.
- **Fiabilité pour 40 ans d'ancienneté :** le modèle n'est **pas fiable**. Les données vont de 0 à 20 ans : c'est de l'**extrapolation**. Le modèle suppose une relation linéaire, qui n'a été vérifiée que sur cet intervalle (en pratique, les salaires plafonnent).
- **Effet d'une augmentation du bruit :** la MSE et la MAE augmentent, et les coefficients appris sont estimés avec moins de précision (ils s'écartent davantage de 600, 1500 et 4000).

# Exercice 3 — Classification des fleurs Iris 🌸

## 1. Préparation des données

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisation ajustée uniquement sur l'entraînement
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("Dimensions :", X_train.shape, y_train.shape)
print("Colonnes :", iris.feature_names)

**Réponses aux questions**

- **Les 4 colonnes de `X_train`** : longueur du sépale, largeur du sépale, longueur du pétale et largeur du pétale (en cm), standardisées.
- **Pourquoi `torch.long` pour les étiquettes ?** `CrossEntropyLoss` attend des **indices de classes entiers** (0, 1, 2) de type `int64`, pas des flottants.
- **Pourquoi ne pas ajuster le `StandardScaler` sur le test ?** Cela ferait fuiter de l'information du jeu de test vers l'entraînement (*data leakage*) et fausserait l'évaluation. On calcule moyenne et écart-type sur l'entraînement seulement, puis on les applique au test avec `transform`.

## 2. Construire le réseau

In [ ]:
model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 3. Entraîner le modèle

In [ ]:
epochs = 200
losses = []

model.train()

for epoch in range(epochs):
    optimizer.zero_grad()                    # remise à zéro des gradients
    logits = model(X_train)                  # scores prédits
    loss = criterion(logits, y_train)        # perte
    loss.backward()                          # gradients
    optimizer.step()                         # mise à jour des paramètres

    losses.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Époque {epoch + 1:3d} | Perte : {loss.item():.4f}")

## 4. Évaluer le classifieur

In [ ]:
model.eval()

with torch.no_grad():
    logits = model(X_test)

    predictions = logits.argmax(dim=1)                     # classe au score maximal
    accuracy = (predictions == y_test).float().mean()      # proportion correcte

print(f"Accuracy sur le test : {accuracy.item():.1%}")

## 5. Autres questions

In [ ]:
# --- Courbe de perte
plt.plot(losses)
plt.xlabel("Époque")
plt.ylabel("Perte (CrossEntropy)")
plt.title("Iris — évolution de la perte")
plt.grid(alpha=0.3)
plt.show()

# --- Matrice de confusion
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test.numpy(), predictions.numpy())
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(cmap="Blues")
plt.title("Iris — matrice de confusion")
plt.show()

In [ ]:
# --- Comparaison avec un modèle sans couche cachée : nn.Linear(4, 3)
def entrainer_iris(modele, epochs=200, lr=0.01):
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(modele.parameters(), lr=lr)
    modele.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = crit(modele(X_train), y_train)
        loss.backward()
        opt.step()
    modele.eval()
    with torch.no_grad():
        return (modele(X_test).argmax(dim=1) == y_test).float().mean().item()

torch.manual_seed(42)
acc_lineaire = entrainer_iris(nn.Linear(4, 3))

torch.manual_seed(42)
acc_cache = entrainer_iris(nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3)))

print(f"Modèle linéaire  nn.Linear(4, 3)        : {acc_lineaire:.1%}")
print(f"Réseau avec couche cachée (4-16-3)      : {acc_cache:.1%}")

In [ ]:
# --- Prédire l'espèce d'une nouvelle fleur
nouvelle_fleur = scaler.transform([[5.1, 3.5, 1.4, 0.2]])
nouvelle_fleur = torch.tensor(nouvelle_fleur, dtype=torch.float32)

model.eval()
with torch.no_grad():
    probas = torch.softmax(model(nouvelle_fleur), dim=1)[0]
    classe = probas.argmax().item()

print("Espèce prédite :", iris.target_names[classe])
print("Probabilités   :", {iris.target_names[i]: round(p.item(), 3) for i, p in enumerate(probas)})

**Remarque :** Iris est presque linéairement séparable, donc le modèle linéaire donne souvent un résultat très proche du réseau avec couche cachée. La couche cachée devient vraiment utile quand les classes ne sont pas séparables par des droites. Les mesures `[5.1, 3.5, 1.4, 0.2]` correspondent à une **setosa**.

# Exercice 2 — Classifier les chiffres MNIST (MLP)

In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Fonctions utilitaires (réutilisées pour l'exercice 4)

In [ ]:
def entrainer(model, train_loader, criterion, optimizer, epochs):
    """Boucle d'entraînement par mini-lots. Retourne l'historique de la perte."""
    historique = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()                   # 1. remise à zéro des gradients
            logits = model(images)                  # 2. scores
            loss = criterion(logits, labels)        # 3. perte
            loss.backward()                         # 4. gradients
            optimizer.step()                        # 5. mise à jour

            total_loss += loss.item() * images.size(0)

        moyenne_perte = total_loss / len(train_loader.dataset)
        historique.append(moyenne_perte)
        print(f"Époque {epoch + 1}/{epochs} | Perte : {moyenne_perte:.4f}")
    return historique


def predire(model, loader):
    """Retourne images, vraies classes et classes prédites sur tout le jeu."""
    model.eval()
    toutes_images, tous_labels, toutes_preds = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            predicted_classes = logits.argmax(dim=1)
            toutes_images.append(images)
            tous_labels.append(labels)
            toutes_preds.append(predicted_classes.cpu())
    return torch.cat(toutes_images), torch.cat(tous_labels), torch.cat(toutes_preds)


def afficher_metriques(y_true, y_pred):
    y_true, y_pred = y_true.numpy(), y_pred.numpy()
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred, average='macro'):.4f}  (macro)")
    print(f"Recall    : {recall_score(y_true, y_pred, average='macro'):.4f}  (macro)")
    print(f"F1-score  : {f1_score(y_true, y_pred, average='macro'):.4f}  (macro)")


def afficher_confusion(y_true, y_pred, titre):
    cm = confusion_matrix(y_true.numpy(), y_pred.numpy())
    ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(cmap="Blues", values_format="d")
    plt.title(titre)
    plt.show()


def afficher_erreurs(images, labels, preds, n=10, mean=0.0, std=1.0):
    """Affiche n images mal classées (annule la normalisation si mean/std fournis)."""
    idx = (preds != labels).nonzero(as_tuple=True)[0][:n]
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), idx):
        img = images[i, 0] * std + mean
        ax.imshow(img.numpy(), cmap="gray")
        ax.set_title(f"Vrai : {labels[i].item()} | Prédit : {preds[i].item()}")
    plt.tight_layout()
    plt.show()

## Entraînement (5 époques)

In [ ]:
historique_mlp = entrainer(model, train_loader, criterion, optimizer, epochs=5)

## Évaluation sur le jeu de test

In [ ]:
images_t, labels_t, preds_t = predire(model, test_loader)

afficher_metriques(labels_t, preds_t)

In [ ]:
afficher_confusion(labels_t, preds_t, "MNIST (MLP) — matrice de confusion")

In [ ]:
afficher_erreurs(images_t, labels_t, preds_t, n=10)

## Sauvegarder et recharger le modèle

In [ ]:
torch.save(model.state_dict(), "mnist_mlp.pth")
print("Modèle sauvegardé dans mnist_mlp.pth")

# On recrée la même architecture, puis on charge les poids : aucun ré-entraînement nécessaire
model_recharge = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model_recharge.load_state_dict(torch.load("mnist_mlp.pth", map_location=device, weights_only=True))
model_recharge.eval()

_, _, preds_recharge = predire(model_recharge, test_loader)
print("Prédictions identiques au modèle d'origine :", torch.equal(preds_t, preds_recharge))
print(f"Accuracy du modèle rechargé : {(preds_recharge == labels_t).float().mean().item():.2%}")

# Exercice 4 — MNIST avec un CNN

## 1. Chargement des données

In [ ]:
torch.manual_seed(42)

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

images, labels = next(iter(train_loader))

print("Images :", images.shape)  # [64, 1, 28, 28]
print("Labels :", labels.shape)  # [64]

**Réponses aux questions**

- **Les 4 dimensions de `images` `[64, 1, 28, 28]`** : taille du lot (64 images), nombre de canaux (1 car niveaux de gris), hauteur (28 px) et largeur (28 px).
- **Pourquoi `shuffle=True` ?** Pour que les mini-lots changent à chaque époque et ne suivent pas l'ordre du jeu de données. Cela évite que le modèle apprenne des biais d'ordre (par exemple des lots remplis d'un seul chiffre) et rend la descente de gradient plus stable.

## 2. Visualiser quelques images

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(12, 3))

for i, ax in enumerate(axes):
    # Annuler la normalisation pour l'affichage
    image = images[i, 0] * 0.3081 + 0.1307

    ax.imshow(image.numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Classe : {labels[i].item()}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Construire le CNN

In [ ]:
class MNISTClassifier(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),   # 1x28x28 -> 16x28x28
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),                                           # -> 16x14x14

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),  # -> 32x14x14
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)                                            # -> 32x7x7
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=dropout),   # dropout=0.0 -> aucun effet (modèle de base)
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = MNISTClassifier().to(device)

# Vérifier la forme de sortie
with torch.no_grad():
    output = model(images.to(device))

assert output.shape == (images.size(0), 10)
print("Sortie :", output.shape)

**Question : pourquoi la première couche linéaire reçoit-elle 32 × 7 × 7 valeurs ?**

L'image 28 × 28 passe par une convolution avec `padding=1` et `kernel_size=3`, qui conserve la taille, puis par un `MaxPool2d(2)` qui la divise par 2 : 28 → 14. La même opération donne 14 → 7. La seconde convolution produit **32 canaux**, donc la sortie de `features` a la forme `32 × 7 × 7`, soit **1568 valeurs** une fois aplatie par `Flatten`.

## 4. Entraîner le réseau

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 3
loss_history = entrainer(model, train_loader, criterion, optimizer, epochs)

## 5. Évaluer sur le jeu de test

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)

        # Classe ayant le score le plus élevé
        predictions = logits.argmax(dim=1)

        # Compter les bonnes prédictions
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Accuracy sur le test : {accuracy:.2%}")

## Autres questions

### Images mal classées

In [ ]:
images_t, labels_t, preds_t = predire(model, test_loader)
afficher_erreurs(images_t, labels_t, preds_t, n=10, mean=MNIST_MEAN, std=MNIST_STD)

### Ajout d'une couche Dropout avant la dernière couche linéaire

In [ ]:
torch.manual_seed(42)
model_dropout = MNISTClassifier(dropout=0.3).to(device)

optimizer_d = torch.optim.Adam(model_dropout.parameters(), lr=0.001)
loss_history_d = entrainer(model_dropout, train_loader, criterion, optimizer_d, epochs=3)

_, _, preds_d = predire(model_dropout, test_loader)
acc_d = (preds_d == labels_t).float().mean().item()
print(f"\nAccuracy sans Dropout : {accuracy:.2%}")
print(f"Accuracy avec Dropout : {acc_d:.2%}")

`Dropout(0.3)` désactive aléatoirement 30 % des neurones pendant l'entraînement (`model.train()`) pour limiter le surapprentissage, et reste inactif en évaluation (`model.eval()`). Sur MNIST avec seulement 3 époques, le gain est faible, voire nul : le modèle n'a pas encore surappris. L'effet devient visible avec plus d'époques ou un modèle plus gros.

### Sauvegarde des paramètres

In [ ]:
torch.save(model.state_dict(), "mnist_cnn.pth")
print("Paramètres sauvegardés dans mnist_cnn.pth")

# Rechargement
model_cnn_recharge = MNISTClassifier().to(device)
model_cnn_recharge.load_state_dict(torch.load("mnist_cnn.pth", map_location=device, weights_only=True))
model_cnn_recharge.eval()

_, _, preds_r = predire(model_cnn_recharge, test_loader)
print("Prédictions identiques :", torch.equal(preds_t, preds_r))